In [34]:
import torch
from datasets import load_dataset


from tqdm import tqdm
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

import math
import regex as re
import tiktoken

import os

current_folder = os.getcwd()
print(current_folder)



ds = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    cache_dir="TinyStories"
)
print(ds)
import time

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import torch.nn as nn
from torch.nn import functional as F

import matplotlib.pyplot as plt


CUDA Available: True
/data/anindra/LLMmodel
Dataset({
    features: ['text'],
    num_rows: 2119719
})


In [11]:
ds.num_rows

2119719

In [12]:
print("DS Rows: ", ds.num_rows/100)

data_range = math.floor((ds.num_rows)/100)

# for i in range(data_range):
#     if "text" not in ds[i]:
#         print(ds[i])
#     if i % (math.floor(data_range/25)) == 0:
#         print(f"Row: {i}/{data_range}")

DS Rows:  21197.19


In [13]:
longest = 0
word = ""
tokens = []
infinity = math.inf

for i in range(data_range):
    if "text" not in ds[i]:
        print(ds[i])
    else: 
        # word = ds[i]["text"]
        # print(map(int, word.encode("utf-8")))
        # tokens.append(map(int, word.encode("utf-8")))
        
        
        if len(ds[i]["text"]) > longest:
            longest = len(ds[i]["text"])
            word = ds[i]["text"]   
        if len(ds[i]["text"]) < infinity:
            infinity = len(ds[i]["text"])
        

# print(longest)

# print(word.encode("utf-8"))

print(f"Shortest Word: {infinity}")
longest_tokens = list(map(int, word.encode("utf-8")))
longest_tokens


Shortest Word: 205


[84,
 111,
 109,
 32,
 97,
 110,
 100,
 32,
 76,
 105,
 108,
 121,
 32,
 119,
 101,
 114,
 101,
 32,
 116,
 119,
 105,
 110,
 115,
 32,
 119,
 104,
 111,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 112,
 108,
 97,
 121,
 32,
 119,
 105,
 116,
 104,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 84,
 104,
 101,
 121,
 32,
 104,
 97,
 100,
 32,
 109,
 97,
 110,
 121,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 32,
 111,
 102,
 32,
 100,
 105,
 102,
 102,
 101,
 114,
 101,
 110,
 116,
 32,
 99,
 111,
 108,
 111,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 115,
 104,
 97,
 112,
 101,
 115,
 46,
 32,
 84,
 104,
 101,
 121,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 98,
 117,
 105,
 108,
 100,
 32,
 116,
 111,
 119,
 101,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 104,
 111,
 117,
 115,
 101,
 115,
 32,
 97,
 110,
 100,
 32,
 99,
 97,
 114,
 115,
 32,
 119,
 105,
 116,
 104,
 32,
 116,
 104,
 101,
 105,
 114,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 83,
 111,
 1

In [14]:

def gets_stats(get_token):
    count = {}

    for i in range(len(get_token) - 1):
        if tuple(get_token[i:i + 2]) not in count.keys():
            count[tuple(get_token[i:i + 2])] = 1
        else:
            count[tuple(get_token[i:i + 2])] += 1
    return count
            
k = gets_stats(longest_tokens)      
gets_stats("Hello World")
# print(sorted(((v,k) for k,v in k.items()), reverse = True))



{('H', 'e'): 1,
 ('e', 'l'): 1,
 ('l', 'l'): 1,
 ('l', 'o'): 1,
 ('o', ' '): 1,
 (' ', 'W'): 1,
 ('W', 'o'): 1,
 ('o', 'r'): 1,
 ('r', 'l'): 1,
 ('l', 'd'): 1}

In [15]:
top_pairs = max(k, key=k.get)
top_pairs



(104, 101)

In [16]:
def merge(ids, pairs, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pairs[0] and ids[i + 1] == pairs[1]:
            newids.append(idx)
            i+=2
        else:
            newids.append(ids[i])
            i += 1
    return newids

        
tokens2 = merge(longest_tokens, top_pairs, 256) 
print(f"Original len: {len(longest_tokens)}, Merge: {len(tokens2)} ")
tokens2

Original len: 4540, Merge: 4320 


[84,
 111,
 109,
 32,
 97,
 110,
 100,
 32,
 76,
 105,
 108,
 121,
 32,
 119,
 101,
 114,
 101,
 32,
 116,
 119,
 105,
 110,
 115,
 32,
 119,
 104,
 111,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 112,
 108,
 97,
 121,
 32,
 119,
 105,
 116,
 104,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 84,
 256,
 121,
 32,
 104,
 97,
 100,
 32,
 109,
 97,
 110,
 121,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 32,
 111,
 102,
 32,
 100,
 105,
 102,
 102,
 101,
 114,
 101,
 110,
 116,
 32,
 99,
 111,
 108,
 111,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 115,
 104,
 97,
 112,
 101,
 115,
 46,
 32,
 84,
 256,
 121,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 98,
 117,
 105,
 108,
 100,
 32,
 116,
 111,
 119,
 101,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 104,
 111,
 117,
 115,
 101,
 115,
 32,
 97,
 110,
 100,
 32,
 99,
 97,
 114,
 115,
 32,
 119,
 105,
 116,
 104,
 32,
 116,
 256,
 105,
 114,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 83,
 111,
 109,
 101,
 116,
 1

In [17]:

# ids = list(tokens)
# print(ids)
# print(len(ids))


num_merge_2 = 20

merges_2 = {}

idx = 256
for j in range(math.floor(data_range/4)):
    word = ds[j]["text"]
    tokens = list(map(int, word.encode("utf-8")))
    ids = list(tokens)
    for i in range(num_merge_2):
        stats = gets_stats(ids)
        top_pairs = max(stats, key=stats.get)
        if top_pairs in merges_2:
            continue
        idx += 1
        ids = merge(ids, top_pairs, idx)
        merges_2[top_pairs] = idx
        
    if j % (math.floor(data_range/25)) == 0:
        print(f"Row: {j}/{math.floor(data_range/4)}")
        

vocab_size = 276
num_merge = vocab_size - 256
print("Part2")
idx = 256
merges = {}
ids = list(longest_tokens)
for i in range(num_merge):
    stats = gets_stats(ids)
    top_pairs = max(stats, key=stats.get)
    idx += 1
    ids = merge(ids, top_pairs, idx)
    merges[top_pairs] = idx




Row: 0/5299
Row: 847/5299
Row: 1694/5299
Row: 2541/5299
Row: 3388/5299
Row: 4235/5299
Row: 5082/5299
Part2


In [23]:
print(f"new merges: {len(merges)}, old merges: {len(merges_2)}")

print(merges)

print()

print(merges_2)


new merges: 20, old merges: 42
{(104, 101): 257, (32, 116): 258, (100, 32): 259, (258, 257): 260, (46, 32): 261, (121, 32): 262, (97, 110): 263, (258, 111): 264, (32, 98): 265, (101, 32): 266, (100, 260): 267, (116, 32): 268, (83, 257): 269, (115, 32): 270, (101, 114): 271, (104, 97): 272, (261, 269): 273, (263, 259): 274, (111, 107): 275, (257, 114): 276}

{(104, 101): 257, (32, 115): 258, (32, 116): 259, (101, 100): 260, (258, 104): 261, (97, 110): 262, (101, 32): 263, (32, 119): 264, (259, 257): 265, (257, 114): 266, (105, 116): 267, (32, 110): 268, (32, 102): 269, (261, 97): 270, (270, 114): 271, (262, 100): 272, (105, 108): 273, (44, 32): 274, (108, 263): 275, (105, 114): 276, (100, 32): 277, (116, 32): 278, (32, 97): 279, (32, 104): 280, (115, 32): 281, (121, 32): 282, (116, 104): 283, (110, 100): 284, (97, 284): 285, (114, 285): 286, (101, 114): 287, (46, 32): 288, (32, 98): 289, (32, 112): 290, (195, 162): 291, (291, 226): 292, (292, 130): 293, (293, 172): 294, (105, 110): 295,

In [24]:
def encode(text, merge_method):
    tokens = list(text.encode("utf-8"))
    while True:
        stats = gets_stats(tokens)
        # print(stats)
        pair = min(stats, key=lambda p: merge_method.get(p, float("inf")))
        if pair not in merge_method:
            # print("pair not in merges")
            break # nothing else can be paired
        idx = merge_method[pair]
        tokens = merge(tokens, pair, idx) 
    return tokens



print(encode("Hello World", merges))
print(encode("Hello World", merges_2))

[72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100]
[72, 101, 297, 111, 32, 87, 298, 108, 100]


In [25]:
vocab = {idx: bytes([idx]) for idx in range(256)}
# print(vocab)

for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
    
vocab_2 = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges_2.items():
    vocab_2[idx] = vocab_2[p0] + vocab_2[p1]


# print(vocab)

def decode(ids, vocab_method):
    tokens = b"".join(vocab_method[idx] for idx in ids)
    text = tokens.decode("utf-8", errors ="replace")
    return text

print(decode(ids, vocab))
print(decode(ids, vocab_2))


Tom and Lily were twins who liked to play with blocks. They had many blocks of different colors and shapes. They liked to build towers and houses and cars with their blocks. Sometimes they shared their blocks, and sometimes they fought over them.

One day, Tom wanted to build a big tower with all the blocks. He did not want to share with Lily. He took all the blocks and put them in a corner. Lily was sad and angry. She wanted to play with some blocks too. She asked Tom to give her some blocks, but Tom said no. He said they were all his blocks.

Lily waited for Tom to finish his tower. She hoped he would let her play with it when he was done. But Tom took a long time to complete his tower. He wanted to make it very tall and strong. He added more and more blocks to his tower. Lily got bored and lonely. She looked around for something else to play with.

She saw a book on the shelf. It was a book of news. It had pictures and words about what was happening in the world. Lily liked to look 

In [26]:
print(decode(encode("Hello World", merges), vocab))
valText = "I cannot express enough how grateful I am for your incredible tutorials. As a 44-year-old South Sudanese individual with three children and a full-time job, recently relocated from the UK to the US, my days are filled to the brim with responsibilities. However, I have made it a priority to utilize every ounce of my free time to follow your series of lectures. I must say, I have never been as excited about learning a new topic as I am now. Your clear explanations and engaging content have ignited a passion for learning within me that I never knew existed. Thank you from the bottom of my heart for all that you do."
valText2 = decode(encode(valText, merges), vocab)
print(valText == valText2)


print(decode(encode("Hello World", merges_2), vocab_2))
valText = "I cannot express enough how grateful I am for your incredible tutorials. As a 44-year-old South Sudanese individual with three children and a full-time job, recently relocated from the UK to the US, my days are filled to the brim with responsibilities. However, I have made it a priority to utilize every ounce of my free time to follow your series of lectures. I must say, I have never been as excited about learning a new topic as I am now. Your clear explanations and engaging content have ignited a passion for learning within me that I never knew existed. Thank you from the bottom of my heart for all that you do."
valText2 = decode(encode(valText, merges_2), vocab_2)
print(valText == valText2)

Hello World
True
Hello World
True


In [27]:
print(encode("Hello World", merges))
print(encode("Hello World", merges_2))

[72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100]
[72, 101, 297, 111, 32, 87, 298, 108, 100]


In [28]:
decode([72, 101, 108, 108, 111, 32, 87, 111, 114,108, 100], vocab)
decode([72, 101, 297, 111, 32, 87, 298, 108, 100], vocab_2)

'Hello World'

In [36]:
count = 0 

divisble = 250
total_rows = math.floor(ds.num_rows/divisble)


start_time = time.perf_counter()


for i in tqdm(range(total_rows)):
    text = ds[i]["text"]
    if decode(encode(text, merges), vocab) == text:
        count += 1
    if i % total_rows == 0:
        print(f"Row {i}/{total_rows}")

print(f"Accuracy: {(count/total_rows)*100}%")


end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.6f} seconds")



# 1. Record the starting time

count = 0

start_time = time.perf_counter()


for i in tqdm(range(total_rows)):
    text = ds[i]["text"]
    if decode(encode(text, merges_2), vocab_2) == text:
        count += 1
    if i % total_rows == 0:
        print(f"Row {i}/{total_rows}")

print(f"Accuracy: {(count/total_rows)*100}%")


end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.6f} seconds")


  1%|▉                                                                             | 99/8478 [00:00<00:29, 283.10it/s]

Row 0/8478


100%|████████████████████████████████████████████████████████████████████████████| 8478/8478 [00:33<00:00, 250.97it/s]


Accuracy: 100.0%
Execution time: 33.783772 seconds


  0%|▎                                                                             | 40/8478 [00:00<00:42, 199.51it/s]

Row 0/8478


100%|████████████████████████████████████████████████████████████████████████████| 8478/8478 [00:55<00:00, 153.09it/s]

Accuracy: 100.0%
Execution time: 55.382002 seconds


In [97]:
class LLM_Dataset(Dataset):
    def __init__(self, text_length, encode, decode, merge_method, vocab_method, device):
        self.device = device
        self.encode = encode
        self.decode = decode
        self.text_length = text_length
        self.merge = merge_method
        self.vocab = vocab_method
        
    def __getitem__(self, idx):
        text = ds[idx]["text"]  
        text_encode = self.encode(text, self.merge)
        
        if len(text_encode) > self.text_length:
            x_text_encode = text_encode[:self.text_length]
            y_text_encode = text_encode[1:self.text_length + 1]
        else: 
            x_text_encode = text_encode + [0] * (self.text_length - len(text_encode))
            y_text_encode = text_encode + [0] * (self.text_length - len(text_encode) + 1)

        return torch.tensor(x_text_encode).to(self.device), torch.tensor(y_text_encode).to(self.device)
    def __len__(self):
        return ds.num_rows

set_device = "cuda" if torch.cuda.is_available() else "cpu"
print(set_device)
LLM = LLM_Dataset(200, encode, decode, merges, vocab, set_device)


x, y = LLM[32]


print(f"x:{x.device},{x}")
print(f"y:{y.device}, {y}")


cuda
x:cuda:0,tensor([ 79, 110,  99, 266, 117, 112, 111, 110,  32,  97, 258, 105, 109, 101,
         44, 260, 114, 266, 119,  97, 270,  97,  32, 108, 105, 116, 116, 108,
        101, 265, 111,  97, 116, 261,  84, 257, 265, 111,  97, 268, 119,  97,
        115, 265, 108, 117, 266, 274, 105, 268, 108, 105, 107, 101, 100, 264,
         32, 102, 108, 111,  97, 268, 111, 110, 260,  32, 119,  97, 116, 271,
        261,  79, 110, 266, 100,  97, 121,  44, 260,  32, 115, 117, 110,  32,
        119,  97, 270, 118, 271, 262, 104, 111, 268, 263, 267,  32, 119,  97,
        116, 271, 265, 101, 103, 263, 264,  32, 100, 114, 262, 117, 112, 261,
         84, 257, 265, 111,  97, 268, 119,  97, 270, 115,  97, 259,  98, 101,
         99,  97, 117, 115, 266, 105, 268,  99, 111, 117, 108, 259, 110, 111,
        268, 102, 108, 111,  97, 268, 263, 121, 109, 111, 114, 101,  46,  10,
         10,  65, 265, 105, 114, 259, 115,  97, 119, 260, 265, 111,  97, 268,
        274,  97, 115, 107, 101, 100,  44,  32,  3

In [98]:
train_divisible = 100

train_set = math.floor(math.floor(ds.num_rows/train_divisible)*0.8)
print(f"Trainin Set: {train_set}")


Trainin Set: 16957


In [99]:
batch_size = 4
# block_size = len(merges) + 256
block_size = 200
num_workers = 0

print(block_size)
print

LLM_dataset = LLM_Dataset(block_size, encode, decode, merges, vocab, set_device)
train_dataloader = DataLoader(LLM_dataset, batch_size = batch_size, shuffle=False, num_workers = num_workers)
# batch_features, batch_labels = len(train_dataloader)

200


In [100]:
for batch in train_dataloader:
    x,y = batch
    break

print(f"X: {x}\n")
print(f"Y: {y}\n")
# print(f"Y: {y}")


X: tensor([[ 79, 110, 266, 100,  97, 121,  44,  32,  97,  32, 108, 105, 116, 116,
         108, 266, 103, 105, 114, 108,  32, 110,  97, 109, 101, 259,  76, 105,
         108, 262, 102, 111, 117, 110, 259,  97,  32, 110, 101, 101, 100, 108,
         266, 105, 110,  32, 276,  32, 114, 111, 111, 109, 273,  32, 107, 110,
         101, 119,  32, 105, 268, 119,  97, 270, 100, 105, 102, 102, 105,  99,
         117, 108, 116, 264,  32, 112, 108,  97, 262, 119, 105, 116, 104,  32,
         105, 116, 265, 101,  99,  97, 117, 115, 266, 105, 268, 119,  97, 270,
         115, 272, 114, 112, 261,  76, 105, 108, 262, 119, 263, 116, 101, 100,
         264,  32, 115, 272, 114, 101, 260,  32, 110, 101, 101, 100, 108, 266,
         119, 105, 116, 104,  32, 276,  32, 109, 111, 109,  44,  32, 115, 111,
          32, 115, 257,  32,  99, 111, 117, 108, 259, 115, 101, 119,  32,  97,
         265, 117, 116, 116, 111, 110,  32, 111, 110,  32, 276,  32, 115, 104,
         105, 114, 116,  46,  10,  10,  76, 105, 

In [101]:
print(x.shape)
print(y.shape)

torch.Size([4, 200])
torch.Size([4, 200])


In [102]:
torch.manual_seed(1337)

class BigramLangaugeModel(nn.Module):
    def __init__(self, vocab_input):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_input, vocab_input)
    def forward(self, idx, targets=None):
        
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else: 
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx


m = BigramLangaugeModel(vocab_input=max(vocab.keys()) + 1)
m.to(set_device)
print(f"X: {x.device}")
print(f"Y: {y.device}")
logits, loss = m(x,y)

print(f"logits.shape: {logits.shape}")
print(f"loss: {loss}")


idx = torch.zeros((1,1), dtype = torch.long).to(set_device)
output = m.generate(idx, torch.tensor(100))[0].tolist()

print(f'generate output: {output}')

print(f"vocab: {len(vocab)}")
print(f"vocab 2: {len(vocab_2)}")

r = decode(output, vocab_2)
print(r)
# print(logits.shape) 

# print(loss.shape)


X: cuda:0
Y: cuda:0
logits.shape: torch.Size([800, 277])
loss: 6.139162540435791
generate output: [0, 260, 101, 44, 134, 62, 220, 125, 49, 80, 101, 153, 130, 255, 41, 142, 93, 139, 6, 38, 67, 276, 178, 24, 221, 239, 6, 130, 145, 103, 23, 189, 150, 147, 208, 28, 145, 271, 251, 109, 226, 40, 185, 178, 59, 87, 107, 119, 36, 31, 136, 233, 27, 7, 9, 155, 114, 118, 55, 90, 123, 92, 121, 211, 125, 0, 262, 36, 234, 24, 114, 179, 236, 18, 132, 80, 163, 273, 224, 122, 55, 243, 235, 259, 244, 60, 8, 58, 88, 247, 152, 1, 42, 36, 9, 196, 31, 264, 39, 186, 43]
vocab: 276
vocab 2: 298
 ede,�>�}1Pe���)�]�&Cir�����g����� shar�m�(��;Wkw$��	�rv7Z{\y�} an$�r���P�il�z7�� t�:X��*$	� w'�+


In [103]:

learning_rate = 1e-3
optimizer = torch.optim.AdamW(m.parameters(), lr = learning_rate)


In [108]:
loop = 10000
for steps in tqdm(range(loop)):
    x, y = next(iter(train_dataloader))

    logits, loss = m(x,y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % loop/10 == 0:
        print(f"Loss {steps}/{loop}: {loss.item()}")



  0%|                                                                              | 15/10000 [00:00<02:22, 70.14it/s]

Loss 0/10000: 3.323026180267334


100%|███████████████████████████████████████████████████████████████████████████| 10000/10000 [02:22<00:00, 70.12it/s]


In [109]:
print(decode(m.generate(idx, torch.tensor(100))[0].tolist(), vocab_2))


 E�=� w thein nen cir sandred w sharan, I ala theee shGo,edryoreherdsh shBepo tfitep shDomil da theiousay srke tLin sh" he omofoonehers shThe 


In [110]:
!nvidia-smi

Sun Jun 14 15:07:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:27:00.0 Off |                    0 |
| N/A   38C    P0             82W /  350W |     511MiB /  46068MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [135]:
torch.manual_seed(1337)
B,T, C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [140]:
x

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])

In [138]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev,0)
xbow

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [139]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ---> (B, T ,C)
xbow2

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [ ]:
torch.allclose(xbow, xbow2)

In [134]:
xbow, xbow2

(tensor([[[ 0.1808, -0.0700],
          [-0.0894, -0.4926],
          [ 0.1490, -0.3199],
          [ 0.3504, -0.2238],
          [ 0.3525,  0.0545],
          [ 0.0688, -0.0396],
          [ 0.0927, -0.0682],
          [-0.0341,  0.1332]],
 
         [[ 1.3488, -0.1396],
          [ 0.8173,  0.4127],
          [-0.1342,  0.4395],
          [ 0.2711,  0.4774],
          [ 0.2421,  0.0694],
          [ 0.0084,  0.0020],
          [ 0.0712, -0.1128],
          [ 0.2527,  0.2149]],
 
         [[-0.6631, -0.2513],
          [ 0.1735, -0.0649],
          [ 0.1685,  0.3348],
          [-0.1621,  0.1765],
          [-0.2312, -0.0436],
          [-0.1015, -0.2855],
          [-0.2593, -0.1630],
          [-0.3015, -0.2293]],
 
         [[ 1.6455, -0.8030],
          [ 1.4985, -0.5395],
          [ 0.4954,  0.3420],
          [ 1.0623, -0.1802],
          [ 1.1401, -0.4462],
          [ 1.0870, -0.4071],
          [ 1.0430, -0.1299],
          [ 1.1138, -0.1641]]]),
 tensor([[[ 0.1808, -0.0700]

In [142]:
tril = torch.tril(torch.ones(T,T))
wei =  torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [129]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [130]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [131]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print("a = ")
print(a)
print("b = ")
print(b)
print("c = ")
print(c)



a = 
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b = 
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c = 
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
